In [1]:
import ee

In [2]:
ee.Initialize(project='citycatalyst')

#### Step 1. Data Processing and Export with GEE

In [3]:
# Porto Alegre bounding box [minLon, minLat, maxLon, maxLat]
poa = ee.Geometry.Rectangle([-51.40, -30.42, -50.82, -29.88])

# --- Global, multitemporal rural-urban classification (classes) ---
degree_urb = ee.Image("JRC/GHSL/P2023A/GHS_SMOD_V2-0/2025").select("smod_code")

img = (degree_urb
       .clip(poa)
       .reproject(crs="EPSG:3857", scale=1000)
       .toInt16())

task = ee.batch.Export.image.toDrive(
    image=img,
    description=f"poa_ghsl_degree_urb_1000m_p2023a_2025",
    folder="gee_exports",
    fileNamePrefix=f"poa_ghsl_degree_urb_1000m_p2023a_2025",
    region=poa,
    scale=1000,
    crs="EPSG:3857",
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)

task.start()
print("Started Drive export:", task.id)

Started Drive export: RELODLU2GYMAJXTN7UOHFIJO


In [ ]:
import time

def wait_for_task(task):
    """Poll task status until complete or failed."""
    while task.active():
        status = task.status()
        state = status.get('state', 'UNKNOWN')
        print(f"  Status: {state}")
        time.sleep(10)
    status = task.status()
    if status.get('state') == 'COMPLETED':
        print("Export finished successfully.")
    else:
        print("Export ended with status:", status)

wait_for_task(task)

#### Step 2. Convert to GOC and Generate Tiles

In [ ]:
# Paths (adjust if your data lives elsewhere)
from pathlib import Path
import subprocess

base = Path("data")
in_tif = base / "poa_ghsl_degree_urb_1000m_p2023a_2025.tif"
out_dir = Path("out/ghsldegree_urb_2025")
cog_tif = out_dir / "poa_ghsl_degree_urb_1000m_p2023a_2025_cog.tif"
tiles_dir = out_dir / "tiles_visual"

out_dir.mkdir(parents=True, exist_ok=True)

In [10]:
# Step 3: Convert to COG (preserve categorical class codes)
subprocess.run([
    "gdal_translate", str(in_tif), str(cog_tif),
    "-of", "COG",
    "-ot", "Int16",
    "-co", "COMPRESS=DEFLATE",
    "-co", "RESAMPLING=NEAREST",
    "-co", "OVERVIEWS=AUTO"
], check=True)
print("Created COG with overviews:", cog_tif)

Input file size is 318, 435
0...10...20...30...40...50...60...70...80...90...100 - done.
Created COG with overviews: out/ghslbuilt_2025/poa_ghsl_built_up_100m_p2023a_2025_cog.tif


In [ ]:
# Colorize GHSL degree of urbanization classes with custom palette
import shutil

colorized_tif = out_dir / "poa_ghsl_degree_urb_colorized.tif"
colors_txt = base / "ghsl_degree_urb_colors.txt"
subprocess.run([
    "gdaldem", "color-relief", "-nearest_color_entry", str(cog_tif), str(colors_txt), str(colorized_tif)
], check=True)

# Preflight: ensure gdal2tiles exists and its Python runtime has numpy
gdal2tiles = shutil.which("gdal2tiles.py")
if not gdal2tiles:
    raise RuntimeError("gdal2tiles.py not found in PATH. Install GDAL first (e.g., `brew install gdal`).")

gdal2tiles_python = None
with open(gdal2tiles, "r", encoding="utf-8", errors="ignore") as f:
    first_line = f.readline().strip()

if first_line.startswith("#!"):
    shebang_parts = first_line[2:].split()
    if shebang_parts:
        if shebang_parts[0].endswith("env") and len(shebang_parts) > 1:
            gdal2tiles_python = shutil.which(shebang_parts[1])
        else:
            gdal2tiles_python = shebang_parts[0]

if not gdal2tiles_python:
    gdal2tiles_python = shutil.which("python3") or shutil.which("python")

if not gdal2tiles_python:
    raise RuntimeError("Could not determine Python interpreter for gdal2tiles.py")

try:
    subprocess.run([gdal2tiles_python, "-c", "import numpy"], check=True, capture_output=True)
except subprocess.CalledProcessError as e:
    raise RuntimeError(
        "Missing numpy in gdal2tiles runtime. Install with: "
        f"`{gdal2tiles_python} -m pip install numpy`"
    ) from e

# Generate XYZ visual tiles
tiles_dir.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py",
    "-r", "near",
    "-z", "8-15",
    "--xyz",
    "-w", "none",
    str(colorized_tif),
    str(tiles_dir)
], check=True)
print("Visual tiles written to:", tiles_dir)

Warning 1: Input dataset has no nodata value. Ignoring 'nv' entry in color palette


0...10...20...30...40...50...60...70...80...90...100 - done.


Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done.
0

Generating Overview Tiles:


...10...20...30...40...50...60...70...80...90...100 - done.
Visual tiles written to: out/ghslbuilt_2025/tiles_visual


#### Export with values 

In [ ]:
# Step 3c: Generate VALUE tiles for client-side hover lookup
# For categorical classes, encode class code directly in R channel (G=0, B=0).
value_encoded_tif = out_dir / "poa_ghsl_degree_urb_value_encoded.tif"
value_colors_txt = base / "ghsl_degree_urb_value_encoding_colors.txt"
value_tiles_dir = out_dir / "tiles_values"

subprocess.run([
    "gdaldem", "color-relief", "-nearest_color_entry", str(cog_tif), str(value_colors_txt), str(value_encoded_tif)
], check=True)

value_tiles_dir.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py",
    "-r", "near",
    "-z", "8-15",
    "--xyz",
    "-w", "none",
    str(value_encoded_tif),
    str(value_tiles_dir)
], check=True)
print("Value tiles written to:", value_tiles_dir)
print("Decode in app with: class_code = R")

0...10...20...30...40...50...60...70...80...90...100 - done.


Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done.
0.

Generating Overview Tiles:


..10...20...30...40...50...60...70...80...90...100 - done.
Value tiles written to: out/ghslbuilt_2025/tiles_values
Decode in app with: value = R + 256*G + 65536*B
